# Problem 3 — Phase 2: TruncatedSVD + LinearSVC

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Prev:** [Phase 1](hierarchical_problem3_phase1_features.ipynb) · **Next:** [Phase 3](hierarchical_problem3_phase3_by_depth.ipynb)

[`svm_tfidf_truncated_svd_linear_edge_factory`](hierarchical_classifier.py): TF-IDF → LSI-style reduction → linear SVM.

Same **`RUN_EXPENSIVE`** gate as Phase 1. With **`VERBOSE_TRAIN = True`**, each full-tree run prints per-edge training progress (`[i/N] parent → child`, `fit: n=…`, and the final summary), same idea as Phase 3 full tree.


In [1]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import load_topic_tree, summary


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — cwd should be Homework 4")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

print("BASE", BASE)
print(summary(tree))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

from hierarchical_classifier import svm_tfidf_truncated_svd_linear_edge_factory
from hierarchical_summary_metrics import comparison_table, train_full_tree_and_summarize


BASE /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4
{'traversal_root': 'Root', 'n_nodes': 117, 'n_leaves': 82, 'n_branching_nodes': 22, 'n_local_classifiers': 22, 'n_binary_edge_classifiers': 103, 'max_branching_factor': 23, 'max_depth_from_root': 5}
train 14465 test 3617


In [2]:
RUN_EXPENSIVE = True
VERBOSE_TRAIN = True  # per-edge fit progress from fit_all_binary_edges (like Phase 3 full tree)

rows_p2: list[dict] = []  # one dict per model from train_full_tree_and_summarize

phase2_specs = [
    ("SVM_SVD500", 500, svm_tfidf_truncated_svd_linear_edge_factory(max_features=MAX_FEATURES, n_components=500)),
    ("SVM_SVD2500", 2500, svm_tfidf_truncated_svd_linear_edge_factory(max_features=MAX_FEATURES, n_components=2500)),
]



In [3]:
if RUN_EXPENSIVE:
    n_specs = len(phase2_specs)
    for i, (name, ncomp, factory) in enumerate(phase2_specs, start=1):
        print("\n" + "=" * 72, flush=True)
        print(
            f"PHASE 2 [{i}/{n_specs}] {name}  (TruncatedSVD n_components={ncomp})",
            flush=True,
        )
        print("=" * 72 + "\n", flush=True)
        t0 = time.perf_counter()
        _, row = train_full_tree_and_summarize(
            name,
            tree,
            mldata,
            factory,
            split=SPLIT,
            verbose=VERBOSE_TRAIN,
            restrict_to_parent_subtree=RESTRICT,
        )
        wall = time.perf_counter() - t0
        row["wall_time_sec"] = wall
        row["n_components"] = ncomp
        rows_p2.append(row)
        print(
            f"\n>>> {name} finished in {wall:.1f}s wall time (metrics on split={SPLIT!r})\n",
            flush=True,
        )
else:
    print("SKIP (RUN_EXPENSIVE=False)")


PHASE 2 [1/2] SVM_SVD500  (TruncatedSVD n_components=500)

[1/103] Root → CCAT  (depth 0)
    fit: n=14465  positives=6540
[2/103] Root → ECAT  (depth 0)
    fit: n=14465  positives=4496
[3/103] Root → GCAT  (depth 0)
    fit: n=14465  positives=6402
[4/103] Root → MCAT  (depth 0)
    fit: n=14465  positives=2035
[5/103] CCAT → C1  (depth 1)


KeyboardInterrupt: 

In [ ]:


df_p2 = comparison_table(rows_p2) if rows_p2 else pd.DataFrame()
if not df_p2.empty:
    display(df_p2.round(4))
    df_p2.to_csv(BASE / "problem3_phase2_results.csv", index=False)
df_p2